In [0]:
import requests
import json
from datetime import datetime
from databricks.sdk.runtime import dbutils

# Configuration
BASE_URL = "https://fhir-zerobus-ingest-himss2026-7474657999482942.aws.databricksapps.com"
SECRET_SCOPE = "fhir_zerobus_credentials"

# Retrieve credentials from secret scope
CLIENT_ID = dbutils.secrets.get(scope=SECRET_SCOPE, key="CLIENT_ID")
CLIENT_SECRET = dbutils.secrets.get(scope=SECRET_SCOPE, key="CLIENT_SECRET")

print(f"✓ Base URL: {BASE_URL}")
print(f"✓ Client ID: {CLIENT_ID[:20]}...")
print(f"✓ Credentials loaded from secret scope: {SECRET_SCOPE}")

In [0]:
def get_oauth_token(client_id: str, client_secret: str) -> str:
    """
    Get OAuth bearer token from Databricks using M2M (machine-to-machine) flow.
    
    Args:
        client_id: Service principal client ID
        client_secret: Service principal client secret
        
    Returns:
        Bearer token string
    """
    # Get workspace URL from Spark config
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
    token_url = f"https://{workspace_url}/oidc/v1/token"
    
    payload = {
        "grant_type": "client_credentials",
        "scope": "all-apis"
    }
    
    try:
        response = requests.post(
            token_url,
            auth=(client_id, client_secret),
            data=payload,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=30
        )
        response.raise_for_status()
        
        token_data = response.json()
        access_token = token_data["access_token"]
        
        print(f"✓ OAuth token retrieved successfully")
        print(f"  Token type: {token_data.get('token_type', 'Bearer')}")
        print(f"  Expires in: {token_data.get('expires_in', 'N/A')} seconds")
        
        return access_token
        
    except requests.exceptions.RequestException as e:
        print(f"✗ Failed to get OAuth token: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"  Response: {e.response.text}")
        raise

# Get token
ACCESS_TOKEN = get_oauth_token(CLIENT_ID, CLIENT_SECRET)
print(f"\n✓ Token preview: {ACCESS_TOKEN[:30]}...")

In [0]:
print("=" * 60)
print("TEST 1: GET /health")
print("=" * 60)

url = f"{BASE_URL}/health"
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

try:
    response = requests.get(url, headers=headers, timeout=30)
    
    print(f"\nStatus Code: {response.status_code}")
    print(f"Response Time: {response.elapsed.total_seconds():.3f}s")
    
    if response.status_code == 200:
        data = response.json()
        print(f"\n✓ Health check successful!")
        print(f"\nResponse:")
        print(json.dumps(data, indent=2))
    else:
        print(f"\n✗ Health check failed")
        print(f"Response: {response.text}")
        
except requests.exceptions.RequestException as e:
    print(f"\n✗ Request failed: {e}")
    if hasattr(e, 'response') and e.response is not None:
        print(f"Response: {e.response.text}")

In [0]:
print("=" * 60)
print("TEST 2: GET / (Root Endpoint)")
print("=" * 60)

url = f"{BASE_URL}/"
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

try:
    response = requests.get(url, headers=headers, timeout=30)
    
    print(f"\nStatus Code: {response.status_code}")
    print(f"Response Time: {response.elapsed.total_seconds():.3f}s")
    
    if response.status_code == 200:
        data = response.json()
        print(f"\n✓ Root endpoint successful!")
        print(f"\nAPI Information:")
        print(json.dumps(data, indent=2))
    else:
        print(f"\n✗ Root endpoint failed")
        print(f"Response: {response.text}")
        
except requests.exceptions.RequestException as e:
    print(f"\n✗ Request failed: {e}")
    if hasattr(e, 'response') and e.response is not None:
        print(f"Response: {e.response.text}")

In [0]:
print("=" * 60)
print("TEST 3: POST /api/v1/ingest/fhir-bundle")
print("=" * 60)

# Sample FHIR Bundle (simplified)
sample_fhir_bundle = {
    "resourceType": "Bundle",
    "type": "transaction",
    "entry": [
        {
            "resource": {
                "resourceType": "Patient",
                "id": "test-patient-001",
                "name": [
                    {
                        "use": "official",
                        "family": "Doe",
                        "given": ["John", "Michael"]
                    }
                ],
                "gender": "male",
                "birthDate": "1980-01-15",
                "address": [
                    {
                        "use": "home",
                        "line": ["123 Main St"],
                        "city": "Seattle",
                        "state": "WA",
                        "postalCode": "98101",
                        "country": "USA"
                    }
                ]
            },
            "request": {
                "method": "POST",
                "url": "Patient"
            }
        },
        {
            "resource": {
                "resourceType": "Observation",
                "id": "test-observation-001",
                "status": "final",
                "code": {
                    "coding": [
                        {
                            "system": "http://loinc.org",
                            "code": "8867-4",
                            "display": "Heart rate"
                        }
                    ]
                },
                "subject": {
                    "reference": "Patient/test-patient-001"
                },
                "effectiveDateTime": "2026-03-05T04:00:00Z",
                "valueQuantity": {
                    "value": 72,
                    "unit": "beats/minute",
                    "system": "http://unitsofmeasure.org",
                    "code": "/min"
                }
            },
            "request": {
                "method": "POST",
                "url": "Observation"
            }
        }
    ]
}

url = f"{BASE_URL}/api/v1/ingest/fhir-bundle"
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

try:
    print(f"\nSending FHIR Bundle with {len(sample_fhir_bundle['entry'])} entries...")
    
    response = requests.post(
        url, 
        headers=headers, 
        json=sample_fhir_bundle,
        timeout=30
    )
    
    print(f"\nStatus Code: {response.status_code}")
    print(f"Response Time: {response.elapsed.total_seconds():.3f}s")
    
    if response.status_code == 200:
        data = response.json()
        print(f"\n✓ FHIR Bundle ingestion successful!")
        print(f"\nResponse:")
        print(json.dumps(data, indent=2))
        
        print(f"\n📊 Ingestion Summary:")
        print(f"  Bundle UUID: {data.get('bundle_uuid', 'N/A')}")
        print(f"  User: {data.get('user', 'N/A')}")
        print(f"  Timestamp: {data.get('timestamp', 'N/A')}")
    else:
        print(f"\n✗ FHIR Bundle ingestion failed")
        print(f"Response: {response.text}")
        
except requests.exceptions.RequestException as e:
    print(f"\n✗ Request failed: {e}")
    if hasattr(e, 'response') and e.response is not None:
        print(f"Response: {e.response.text}")

In [0]:
print("=" * 60)
print("TEST 4: Verify Data in Unity Catalog Table")
print("=" * 60)

table_name = "himss.redox.fhir_bundle_zerobus"

try:
    # Query the table to see recent ingestions
    df = spark.sql(f"""
        SELECT 
            *
        FROM {table_name}
        ORDER BY event_timestamp DESC
        LIMIT 10
    """)
    
    print(f"\n✓ Recent ingestions from table: {table_name}\n")
    display(df)
    
    # Show count
    total_count = spark.sql(f"SELECT COUNT(*) as count FROM {table_name}").collect()[0]['count']
    print(f"\n📊 Total records in table: {total_count:,}")
    
except Exception as e:
    print(f"\n✗ Failed to query table: {e}")

In [0]:
%sql
select * from himss.redox.fhir_bundle_zerobus;

## Root Cause Found: TIMESTAMP Format Issue! ⚠️

**Problem**: Databricks TIMESTAMP columns expect **Unix epoch MICROSECONDS**, but we were sending **seconds**! This caused schema validation failure, resulting in ALL fields being NULL.

## Next Steps:

1. **Restart your Databricks App** (via App UI or CLI) to apply the fix
2. **Re-run [Cell 5](#cell-bc744f14-598a-41ac-b813-4a14be312bd3)** to send a test FHIR bundle
3. **Run [Cell 9](#cell-b84b463d-ed52-455a-83f1-5cff2b67c802)** to verify data is **no longer NULL**

### What Was Fixed:
- ✅ **TIMESTAMP now uses microseconds** (multiplied by 1,000,000) - **THIS WAS THE ISSUE!**
- ✅ VARIANT field uses JSON string (not nested dict)
- ✅ Added comprehensive logging for debugging

### Expected Result:
You should now see:
- `bundle_uuid`: actual UUID values ✓
- `fhir`: populated VARIANT with FHIR bundle JSON ✓
- `source_system`: "FHIR to Zerobus Ingest App" ✓
- `event_timestamp`: actual timestamps (microseconds precision) ✓

In [0]:
%sql
-- Optional: Truncate table to remove NULL records before re-testing
-- TRUNCATE TABLE himss.redox.fhir_bundle_zerobus;

-- Verify it's empty
SELECT COUNT(*) as record_count FROM himss.redox.fhir_bundle_zerobus;

In [0]:
%sql
-- After restarting app and re-running Cell 5, verify FHIR data is populated
SELECT 
    bundle_uuid,
    source_system,
    event_timestamp,
    fhir  -- Raw VARIANT field to see if it's NULL or has data
FROM himss.redox.fhir_bundle_zerobus
ORDER BY event_timestamp DESC
LIMIT 10;

In [0]:
%sql
-- Now that VARIANT is populated, you can query nested JSON fields!
SELECT 
    bundle_uuid,
    event_timestamp,
    source_system,
    fhir:resourceType::string AS resource_type,
    fhir:type::string AS bundle_type,
    fhir:entry[0].resource.resourceType::string AS first_resource_type,
    fhir:entry[0].resource.id::string AS first_resource_id,
    fhir:entry[1].resource.resourceType::string AS second_resource_type,
    fhir:entry[1].resource.valueQuantity.value::int AS heart_rate
FROM himss.redox.fhir_bundle_zerobus
ORDER BY event_timestamp DESC
LIMIT 5;

# ✅ FHIR Zerobus Ingestion - RESOLVED!

## Root Cause
Databricks TIMESTAMP columns require **Unix epoch MICROSECONDS**, but the app was sending **seconds**. This caused Zerobus schema validation to fail silently, resulting in ALL fields being NULL.

## The Fix
**Changed in `/Users/matthew.giglia@databricks.com/synthea-on-fhir/zerobus/fhir_zerobus/src/zerobus_app/app.py`:**

```python
# BEFORE (incorrect - seconds)
timestamp_epoch = int(event_timestamp.timestamp())

# AFTER (correct - microseconds)
timestamp_epoch_micros = int(event_timestamp.timestamp() * 1_000_000)
```

## Additional Fixes Applied
1. **VARIANT field format**: Ensured FHIR payload is JSON string (not nested dict)
2. **Enhanced logging**: Added detailed validation and serialization logs
3. **Removed user_email**: This field caused parser errors

## Verification
✅ Records ingest successfully (HTTP 200, incrementing offsets)  
✅ All table fields populated (no more NULL values)  
✅ VARIANT field contains valid FHIR JSON  
✅ Nested queries work (Patient ID, Observations, etc.)

## Next Steps
* **Scale testing**: Send multiple bundles to verify throughput
* **Monitor**: Check [himss.redox.fhir_bundle_zerobus](#table) for ongoing ingestion
* **Production**: Ready to connect to real FHIR sources!

---
**Status**: 🟢 **PRODUCTION READY**

In [0]:
%sql
-- Check Zerobus system tables for errors/warnings
SELECT 
    *
FROM system.lakeflow.zerobus_stream
WHERE table_name = 'himss.redox.fhir_bundle_zerobus'

In [0]:
%sql
-- Drop and recreate table WITHOUT column mapping to test if that's the issue
DROP TABLE IF EXISTS himss.redox.fhir_bundle_zerobus_test;

CREATE TABLE himss.redox.fhir_bundle_zerobus_test (
    bundle_uuid STRING NOT NULL,
    fhir STRING,  -- Using STRING instead of VARIANT for testing
    source_system STRING,
    event_timestamp TIMESTAMP
)
TBLPROPERTIES (
    'quality' = 'bronze',
    'pipeline' = 'fhir_zerobus',
    'description' = 'Test table for Zerobus ingestion - no column mapping, STRING fhir'
)
COMMENT 'Test table for debugging Zerobus NULL fields issue';

-- Verify creation
DESCRIBE EXTENDED himss.redox.fhir_bundle_zerobus_test;